<a href="https://colab.research.google.com/github/JaberAhmad555/flyrank-ml-internship/blob/main/01_first_look_and_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Run it, then discover a real truth yourself

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JaberAhmad555/flyrank-ml-internship/blob/main/notebooks/01_first_look_and_discovery.ipynb?flush_cache=true)

By the end of this notebook you will have:
1. **Run a real ML pipeline** on real (anonymized) search data and watched a learned model beat a hand-written rule.
2. **Rediscovered a real finding yourself** in ~10 lines of pandas.

No prior ML needed. Everything runs on the small anonymized dataset that ships with this repo — no credentials, no private data.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

#os-Operating System-এর সাথে কাজ করে।
#sys-Python environment-এর information দেয়। যেমন আমরা Colab-এ আছি কিনা check করা।
#subprocess-Python-এর ভিতর থেকে terminal command চালানো যায়।যেমন:git clone ...

IN_COLAB = "google.colab" in sys.modules

# বর্তমানে Python কোন কোন module load করেছে তার list-এর মতো।
# যদি:"google.colab"থাকে, তাহলে:IN_COLAB = True

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
#Repo clone হলে যে folder তৈরি হবে তার নাম।

if IN_COLAB:
    if not os.path.isdir(REPO_DIR): #flyrank-ml-internship-starter folder already আছে?
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

  #  এইটা practically terminal-এ:
# git clone --depth 1 https://github...চালানোর মতো।

# git clone GitHub repo Colab computer-এ copy করে।

# --depth 1 পুরো Git history download করবে না।শুধু latest version নেবে।
# কারণ old commits দরকার নাই।
# check=True Command fail করলে Python যেন silently ignore না করে।Error দেখাবে।


    os.chdir(REPO_DIR)
# chdir = change directory মানে current location পরিবর্তন করে FlyRank repo-এর ভিতরে ঢুকছে।
# Example:
# আগে:/content
# পরে:/content/flyrank-ml-internship-starter
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# requirements.txt-এ যেসব Python library লেখা আছে সব install করো।
#sys.executable যে Python এখন notebook চালাচ্ছে, সেই Python use করো।
# -q quiet mode।অর্থাৎ unnecessary অনেক output দেখাবে না।
# -r requirements.txt requirements.txt file-এর package list follow করো।
else:
    # find the repo root from wherever this kernel started

# Local PC হলে?
# else:মানে যদি Colab না হয়।
# তারপর:while not os.path.isdir("data/raw") and os.getcwd() != "/":
#     os.chdir("..")
# এখানে Python খুঁজছে:data/raw folder।কারণ এটা repo root-এর ভিতর থাকার কথা।ধরেন notebook accidentally এখানে start হয়েছে:
# repo/notebooks/
# তখন:os.chdir("..")
# মানে এক folder পিছিয়ে:repo/ এ চলে আসবে।


    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
# os.getcwd() মানে:Get Current Working Directory বর্তমানে কোন folder-এ আছি সেটা দেয়।


print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
# assert মানে: আমি expect করছি conditionটা True হবে। এখানে check করছে CSV file আছে কিনা।
# থাকলে → continue।না থাকলে → error:
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Run the whole pipeline

This runs `scripts/run_all.py`: prepare features → baseline rule → train 3 models → evaluate → PDF.
It takes ~1 minute on the 30,000-row sample.

In [ ]:
# Watch it work — each of the 5 steps prints live as it runs (~1 minute total).
!{sys.executable} scripts/run_all.py

# ! মানে:এটা Python statement না, terminal command হিসেবে চালাও।



▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)


### What just happened?
The pipeline ranked every page for "refresh review" two ways: a **hand-written rule baseline** and a **learned model**. Let's compare them on **Precision@50** — of the top 50 pages each says to fix first, how many are actually declining?

In [ ]:
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule  Precision@50: {base:.3f}   (~{round(base*50)} of the top 50 right)")
print(f"Random forest      Precision@50: {rf:.3f}   (~{round(rf*50)} of the top 50 right)")
print(f"\nThe learned model roughly {rf/base:.1f}x the rule on this metric.")
print("Validation split used:", res["split_strategy"], "(pages from a client are never in both train and test)")

# প্রথমে:
# import json JSON file পড়ার library।
# res = json.load(open("outputs/model_results.json"))

#open(...)

# file খুলছে: outputs/model_results.json
# json.load(...) JSON content Python dictionary-তে convert করছে।
# অর্থাৎ এখন:res এর ভিতরে model result আছে।

# Conceptually:
# res = {"baseline": {...},"models": {...},"split_strategy": ...}
# তারপর:base = res["baseline"]["baseline_precision_at_50"]
#এখানে dictionary-এর ভিতর থেকে:baseline Precision@50 নিচ্ছে।
# ধরেন:base = 0.24
# তারপর:# rf = res["models"]["random_forest"]["precision_at_50"]
# Random Forest-এর Precision@50 নিচ্ছে।
# ধরেন:
# rf = 0.70 এরপর:
# print(
#     f"Hand-written rule Precision@50: {base:.3f}" )
#এই অংশ:
# round(base*50) ধরেন precision:0.60
# তাহলে:0.60 × 50 = 30
# অর্থাৎ top 50-এর মধ্যে approximately 30 correct।
# একইভাবে: round(rf*50)
# Random Forest-এর correct count।তারপর:rf/base
# এটা model baseline-এর চেয়ে কতগুণ better দেখাচ্ছে।
# ধরেন:
# RF = .72
# Baseline = .24
# তাহলে:
# .72 / .24 = 3
# অর্থাৎ roughly:3x
# শেষ line:
# res["split_strategy"] model evaluation-এ কীভাবে train/test ভাগ হয়েছে সেটা দেখায়।
# Notebook বলছে client-holdout split use হয়েছে।
# Client holdout কেন?
# ধরেন:Client A-এর page
# training-এও দিলেন এবং testing-এও দিলেন।
# তাহলে model already client-এর pattern দেখে ফেলতে পারে।
# এটা unfair হতে পারে।
# তাই:
# Client A → train
# Client B → test

#separation করা হয়।



You just ran a real ML system on real search data and saw a learned ranking beat a fixed rule. Now open `outputs/model_report.md` and skim it — that Markdown report is the *shape* of what your own capstone should produce.

## 2. Discover a real truth yourself

The safest, most satisfying early wins are **things you find in the data** — un-leakable, and they *are* the core lesson. Run the three cells below. Each is ~10 lines of pandas and each overturns a common SEO belief.

Every number is **computed live from the shipped CSV** — nothing is hardcoded.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

### Discovery A — "High search volume means more traffic." Does it?

In [ ]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> keyword search volume barely predicts the traffic a page actually gets.")

# এই line খুব important।
# প্রথমে:
# df["search_volume"]
# search_volume column নেয়।
# তারপর:
# df["impressions_90d"]
# last 90 days-এর impressions column।
# Impression কী?
# Google search-এ আপনার page result হিসেবে দেখানো হয়েছে।
# এটা click না।
# Example:
# আপনার page Google result-এ:
# 1000 times দেখা গেল
# 50 clicks পেল
# তাহলে:
# Impressions = 1000
# Clicks = 50

# .corr()
# Correlation calculate করে।
# Correlation roughly:
# +1 = strong positive relation
#  0 = almost no linear relation
# -1 = opposite relation
# Example:
# Height ↑
# Weight generally ↑
# positive correlation হতে পারে।



### Discovery B — the CTR cliff by position
Click-through rate is not flat: it collapses as you move down the results.

In [ ]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_pos.round(4).to_string())
ctr_by_pos.plot(kind="bar", title="Mean CTR by position tier (impressions >= 100)", ylabel="CTR");


# প্রথম:
# visible = df[df["impressions_90d"] >= 100]
# ভেঙে দেখি।
# df["impressions_90d"] >= 100
# প্রত্যেক row-এর জন্য True/False বানাবে।
# Example:
# 50   → False
# 120  → True
# 500  → True
# তারপর:
# df[...]
# শুধু True row রাখে।
# অর্থাৎ:
# যেসব page অন্তত 100 impressions পেয়েছে শুধু সেগুলো analyse করি।
# কেন?
# খুব low impression হলে CTR unstable হতে পারে।
# Example:
# 1 impression + 1 click:
# CTR = 100%
# কিন্তু সেটা reliable picture না।
# তারপর:
# ctr_by_pos = (
#     visible
#     .groupby("position_tier")["ctr"]
#     .mean()
#     .sort_values(ascending=False)
# )
# এটা ভেঙে বুঝি।
# .groupby("position_tier")
# একই position tier-এর page একসাথে group করে।
# Example:
# Position 1-3 → one group
# Position 4-10 → one group
# Position 11-20 → one group
# ["ctr"]
# প্রত্যেক group থেকে শুধু CTR column নিয়ে কাজ করবে।
# .mean()
# প্রত্যেক group-এর average CTR বের করে।
# .sort_values(ascending=False)
# বড় থেকে ছোট সাজায়।
# False মানে descending।
# তারপর:
# print(ctr_by_pos.round(4).to_string())
# .round(4)
# 4 decimal।
# .to_string()
# clean text আকারে দেখায়।
# তারপর:
# ctr_by_pos.plot(
#     kind="bar",
#     title="Mean CTR by position tier (impressions >= 100)",
#     ylabel="CTR"
# );
# এটা bar chart বানায়।
# kind="bar"
# Bar graph।
# title=
# graph title।
# ylabel="CTR"
# Y-axis-এর নাম।
# শেষের:
# ;

# Jupyter-এর extra text output suppress করে।



### Discovery C — is longer content the lever?
Compare word count for **declining** vs **growing** pages.

In [ ]:
wc = df.groupby("trend_direction")["word_count"].median()
print(wc.round(0).to_string())
print("\n'down' vs 'up' pages have almost the same median word count -> length is not the lever.")

# groupby("trend_direction")
# page group করবে trend অনুযায়ী।
# Possible conceptual groups:
# down
# flat
# up
# ["word_count"]
# word count নিচ্ছে।
# .median()
# এটা mean না।
# Median = মাঝের value।
# Example:100, 200, 300, 400, 10000
# Median:
# 300
# Mean অনেক বেশি হবে 10000-এর জন্য।
# Content length-এর মতো data-তে outlier থাকতে পারে, তাই median useful।
# তারপর:
# print(wc.round(0).to_string())
# nearest whole number-এ word count দেখাচ্ছে।
# তারপর:
# print(
#     "\n'down' vs 'up' pages have almost the same median word count..."
# )


## 3. 🔧 Your turn

Pick **one** of these and write a few lines below. This is your Week-1 discovery — you'll reference it in your Week-1 research-question write-up (on the InternHQ board).

- Redo Discovery A but only for pages with `impressions_90d > 0` — does the correlation change?
- In Discovery B, which `content_type` has the worst CTR *at the same position tier*?
- Find another belief to test: does `content_age_days` relate to `trend_direction`? Does `avg_position` relate to `ctr`?

**Rules:** describe what you observe as *observed / directional* — never "I proved Google's algorithm." Keep client data out of anything you publish.

In [ ]:
# Your discovery here
pages_with_impressions = df[df["impressions_90d"] > 0].copy()
filtered_corr = pages_with_impressions["search_volume"].corr(pages_with_impressions["impressions_90d"])
print("pages with impressions>0",len(pages_with_impressions))
print(f"original correlation : {corr:.3f}")
print(f"filtered correlation : {filtered_corr:.3f}")

After filtering the dataset to include only pages with more than zero impressions, all 30,000 pages remained. The correlation between search volume and impressions was still 0.001, exactly the same as before. Therefore, in this dataset, removing zero-impression pages does not change the observed relationship, and search volume still has almost no linear correlation with impressions.

### Save your work
**Colab:** *File → Save a copy in GitHub* (writes to your own repo — that's your submission) and *File → Save a copy in Drive* (so the session doesn't evaporate).

Next: `02_your_first_readable_model.ipynb` — where the model becomes a rule you can read.